# Chapter 13: Human-in-the-Loop (HITL)

EliteAI Academy

Add human review through confidence gates, moderation, support escalation, policy controls, feedback collection, and approval gates.

## 0. Setup

Run `setup.ps1` from the course folder once, then select **Python (Agentic AI Course)** as the kernel.
The course loads your DeepSeek credentials from `.env`; restart the kernel after editing that file.
The install cell below is optional if setup is already complete. Run the remaining cells in order.

**Teaching check:** explain the pattern, run one example, inspect its output, then try an exercise.
Live model calls use API credits and may produce different answers. Stock prices, weather, CRM records,
email delivery, and approval workflows in these lessons are classroom simulations. Model-generated
research is illustrative and needs independent source verification before factual use.

**Simulation:** model-generated reviewer decisions are not real human approval. No real transactions or deployments occur.


In [ ]:
%pip install -r requirements-lock.txt --quiet

In [ ]:
from course_config import MODEL
import os, json, re, time, random, uuid
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple
from enum import Enum
from datetime import datetime

PROVIDER = "deepseek"

if PROVIDER == "deepseek":
    import anthropic
    _client = anthropic.Anthropic(timeout=60.0, max_retries=2)
    _model  = MODEL
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 512) -> str:
        r = _client.messages.create(thinking={"type": "disabled"}, model=_model, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": prompt}])
        return r.content[0].text.strip()

elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    _gmodel = genai.GenerativeModel("gemini-2.0-flash")
    def call_llm(prompt: str, system: str = "You are a helpful AI assistant.", max_tokens: int = 512) -> str:
        return _gmodel.generate_content(f"{system}\n\n{prompt}").text.strip()

print(f"✅ Provider: {PROVIDER}")
print(call_llm("Reply: HITL READY"))

## 1. Core HITL Infrastructure

In [ ]:
class ConfidenceLevel(Enum):
    HIGH   = "high"
    MEDIUM = "medium"
    LOW    = "low"
    NONE   = "none"

@dataclass
class HITLDecision:
    'Represents a decision point that may require human input.'
    decision_id:   str = field(default_factory=lambda: f"DEC-{uuid.uuid4().hex[:6].upper()}")
    task:          str = ""
    ai_output:     str = ""
    confidence:    ConfidenceLevel = ConfidenceLevel.HIGH
    confidence_score: float = 1.0
    reason:        str = ""
    escalated:     bool = False
    human_input:   Optional[str] = None
    final_output:  Optional[str] = None
    timestamp:     str = field(default_factory=lambda: datetime.now().isoformat()[:19])

    def display(self):
        icon = {ConfidenceLevel.HIGH: "✅", ConfidenceLevel.MEDIUM: "⚠️ ",
                ConfidenceLevel.LOW: "🔴", ConfidenceLevel.NONE: "🚨"}[self.confidence]
        print(f"  {icon} [{self.decision_id}] confidence={self.confidence.value} "
              f"({self.confidence_score:.0%}) | escalated={self.escalated}")
        if self.reason:
            print(f"     Reason: {self.reason}")

class ConfidenceScorer:
    'Scores AI output confidence and determines if human review is needed.'

    THRESHOLDS = {
        ConfidenceLevel.HIGH:   0.85,
        ConfidenceLevel.MEDIUM: 0.65,
        ConfidenceLevel.LOW:    0.40,
    }

    def score(self, task: str, output: str, domain: str = "general") -> Tuple[float, str]:
        raw = call_llm(
            f"""Rate your confidence in this output on a scale of 0.0–1.0.

Task: {task}
Domain: {domain}
Output: {output[:400]}

Consider: factual accuracy, completeness, potential for harm if wrong, ambiguity.
Return ONLY JSON: {{"score": <0.0-1.0>, "reason": "<brief explanation>"}}""",
            system="You are a calibrated confidence evaluator. Be honest about uncertainty."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            return float(data["score"]), data["reason"]
        except:
            return 0.5, "Could not parse confidence score"

    def classify(self, score: float) -> ConfidenceLevel:
        if score >= self.THRESHOLDS[ConfidenceLevel.HIGH]:   return ConfidenceLevel.HIGH
        if score >= self.THRESHOLDS[ConfidenceLevel.MEDIUM]: return ConfidenceLevel.MEDIUM
        if score >= self.THRESHOLDS[ConfidenceLevel.LOW]:    return ConfidenceLevel.LOW
        return ConfidenceLevel.NONE

class HumanReviewQueue:
    'Simulates a human review queue.'
    def __init__(self):
        self.queue:    List[HITLDecision] = []
        self.resolved: List[HITLDecision] = []

    def submit(self, decision: HITLDecision):
        decision.escalated = True
        self.queue.append(decision)
        print(f"  📬 Submitted to human review queue: {decision.decision_id}")

    def simulate_human_review(self, decision: HITLDecision,
                               human_response: str = None) -> HITLDecision:
        'Simulates a human reviewing and responding.'
        if human_response:
            decision.human_input  = human_response
            decision.final_output = human_response
        else:

            decision.human_input = call_llm(
                f"You are a human expert. Review this AI output and provide your expert judgment.\n"
                f"Task: {decision.task}\nAI Output: {decision.ai_output}\n"
                f"Provide a brief expert judgment (approve/modify/reject + reasoning).",
                system="You are a human domain expert. Be direct and decisive."
            )
            decision.final_output = decision.human_input

        self.queue.remove(decision)
        self.resolved.append(decision)
        print(f"  👤 Human reviewed [{decision.decision_id}]: {decision.human_input[:100]}")
        return decision

    def status(self):
        print(f"\n📊 Review Queue: {len(self.queue)} pending | {len(self.resolved)} resolved")

print("✅ HITL infrastructure ready: ConfidenceScorer, HITLDecision, HumanReviewQueue")

## 2. Core HITL Agent — Confidence-Gated Human Review

In [ ]:
class HITLAgent:
    'Agent that autonomously handles high-confidence tasks'
    def __init__(self, role: str, domain: str, review_queue: HumanReviewQueue,
                 scorer: ConfidenceScorer, auto_escalate_threshold: float = 0.65):
        self.role       = role
        self.domain     = domain
        self.queue      = review_queue
        self.scorer     = scorer
        self.threshold  = auto_escalate_threshold
        self.decisions: List[HITLDecision] = []

    def handle(self, task: str, force_review: bool = False) -> HITLDecision:

        ai_output = call_llm(task, system=f"You are a {self.role}. Be precise and thorough.")

        score, reason = self.scorer.score(task, ai_output, self.domain)
        level = self.scorer.classify(score)

        decision = HITLDecision(
            task=task, ai_output=ai_output,
            confidence=level, confidence_score=score, reason=reason
        )
        self.decisions.append(decision)
        decision.display()

        if force_review or level in (ConfidenceLevel.LOW, ConfidenceLevel.NONE):
            print(f"  → Routing to human review (score={score:.2f} < threshold={self.threshold:.2f})")
            self.queue.submit(decision)
            decision = self.queue.simulate_human_review(decision)
        elif level == ConfidenceLevel.MEDIUM:
            print(f"  → Medium confidence — flagging for async review, proceeding with AI output")
            decision.final_output = ai_output
            self.queue.submit(decision)
        else:
            print(f"  → High confidence — autonomous action")
            decision.final_output = ai_output

        return decision

    def summary(self):
        total    = len(self.decisions)
        auto     = sum(1 for d in self.decisions if not d.escalated)
        escalated = total - auto
        avg_conf = sum(d.confidence_score for d in self.decisions) / total if total else 0
        print(f"\n📊 Agent Summary: {total} tasks | {auto} autonomous | "
              f"{escalated} escalated | avg confidence={avg_conf:.0%}")

print("=== CONFIDENCE-GATED HITL AGENT ===")
queue   = HumanReviewQueue()
scorer  = ConfidenceScorer()
medical = HITLAgent(
    role="medical triage assistant",
    domain="healthcare",
    review_queue=queue,
    scorer=scorer,
    auto_escalate_threshold=0.65
)

tasks = [
    "Patient reports mild headache for 2 hours, no other symptoms. What should they do?",
    "65-year-old patient with chest pain, shortness of breath, and left arm numbness. Action?",
    "Child has fever of 38.5°C for 3 days despite paracetamol. Next steps?",
]

for task in tasks:
    print(f"\n📋 Task: {task[:70]}...")
    decision = medical.handle(task)
    print(f"  Final: {decision.final_output[:150]}")

medical.summary()
queue.status()

## 3. Use Case 1 — Content Moderation with HITL

In [ ]:
class ModerationVerdict(Enum):
    APPROVE  = "approve"
    REJECT   = "reject"
    ESCALATE = "escalate"

def ai_moderate(content: str) -> Tuple[ModerationVerdict, float, str]:
    'AI-first moderation pass. Returns (verdict, confidence, reason).'
    raw = call_llm(
        f"""Evaluate this content for policy violations.
Policies: no hate speech, no violence, no adult content, no spam.

Content: \"{content}\"

Return ONLY JSON:
{{"verdict": "approve"|"reject"|"escalate", "confidence": <0.0-1.0>, "reason": "<brief>"}}""",
        system="You are a content moderation AI. Be precise. Escalate ambiguous cases."
    )
    try:
        data    = json.loads(re.sub(r"```[\w]*", "", raw).strip())
        verdict = ModerationVerdict(data["verdict"])
        return verdict, float(data["confidence"]), data["reason"]
    except:
        return ModerationVerdict.ESCALATE, 0.0, "Parse error"

def human_moderate(content: str, ai_reason: str) -> Tuple[ModerationVerdict, str]:
    'Simulated human moderator reviewing an escalated case.'
    decision = call_llm(
        f"""You are a human content moderator with 5 years experience.
The AI flagged this content as ambiguous.

Content: \"{content}\"
AI reasoning: {ai_reason}

Make a final decision: approve or reject? Return JSON:
{{"verdict": "approve"|"reject", "notes": "<your reasoning>"}}""",
        system="You are an experienced human moderator. Policies: ban hate, violence, adult, spam."
    )
    try:
        data = json.loads(re.sub(r"```[\w]*", "", decision).strip())
        return ModerationVerdict(data["verdict"]), data["notes"]
    except:
        return ModerationVerdict.REJECT, "Could not parse human decision"

def moderate_content_pipeline(items: List[str]) -> dict:
    'Full moderation pipeline with AI-first, human escalation.'
    results = {"approved": [], "rejected": [], "human_reviewed": 0}

    for i, content in enumerate(items):
        print(f"\n[Item {i+1}] {content[:60]}...")
        verdict, confidence, reason = ai_moderate(content)
        print(f"  AI: {verdict.value} (confidence={confidence:.0%}) | {reason}")

        if verdict == ModerationVerdict.ESCALATE or confidence < 0.75:

            print(f"  🔴 Escalating to human moderator...")
            verdict, notes = human_moderate(content, reason)
            results["human_reviewed"] += 1
            print(f"  👤 Human: {verdict.value} | {notes}")

        (results["approved"] if verdict == ModerationVerdict.APPROVE
         else results["rejected"]).append(content[:40])

    return results

print("=== CONTENT MODERATION HITL PIPELINE ===")
content_items = [
    "Check out our new summer sale — 50% off all items this weekend!",
    "I hate people who drive slowly, they should be banned from roads.",
    "This political policy is absolutely destroying our country.",
    "Beautiful sunset photo from my garden today 🌅",
]

results = moderate_content_pipeline(content_items)
print(f"\n📊 Results: {len(results['approved'])} approved | "
      f"{len(results['rejected'])} rejected | {results['human_reviewed']} human-reviewed")

## 4. Use Case 2 — Technical Support with Escalation

In [ ]:
class SupportTier(Enum):
    TIER1 = "tier1"
    TIER2 = "tier2"
    TIER3 = "tier3"

@dataclass
class SupportSession:
    session_id:     str = field(default_factory=lambda: f"SESS-{uuid.uuid4().hex[:6].upper()}")
    customer:       str = ""
    issue:          str = ""
    tier:           SupportTier = SupportTier.TIER1
    history:        List[Dict] = field(default_factory=list)
    resolved:       bool = False
    escalation_reason: str = ""

def classify_support_issue(issue: str) -> Tuple[SupportTier, str]:
    'Classify issue complexity and route to appropriate tier.'
    raw = call_llm(
        f"""Classify this customer support issue.

Issue: {issue}

Tiers:
- tier1: routine (FAQ, account basics, simple how-to)
- tier2: intermediate (billing disputes, technical bugs, account issues)
- tier3: complex/sensitive (data breach, legal threats, extreme frustration, unusual edge cases)

Return ONLY JSON: {{"tier": "tier1"|"tier2"|"tier3", "reason": "<brief>"}}""",
        system="You are a support triage classifier. Return only valid JSON."
    )
    try:
        data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
        return SupportTier(data["tier"]), data["reason"]
    except:
        return SupportTier.TIER3, "Classification failed — escalating to be safe"

def tier1_handler(session: SupportSession) -> str:
    'AI chatbot handles routine queries.'
    return call_llm(
        session.issue,
        system="You are a helpful customer support chatbot. Resolve simple issues clearly."
    )

def tier2_handler(session: SupportSession) -> str:
    'AI + simulated tool use for intermediate issues.'

    account_data = {"status": "active", "plan": "Pro", "balance": "$24.99 due"}
    ticket_id    = f"TKT-{random.randint(10000, 99999)}"

    response = call_llm(
        f"Issue: {session.issue}\nAccount data: {account_data}\nTicket ID: {ticket_id}\n"
        f"Resolve this issue using the account data. Create a ticket if needed.",
        system="You are a senior support agent with access to account tools. Be empathetic and precise."
    )
    return f"[Ticket {ticket_id} created] {response}"

def tier3_human_handler(session: SupportSession) -> str:
    'Human specialist handles complex/sensitive issues (simulated).'
    return call_llm(
        f"You are a senior human support specialist.\nCustomer: {session.customer}\n"
        f"Issue: {session.issue}\nEscalation reason: {session.escalation_reason}\n"
        f"Provide a compassionate, expert resolution. You have full authority to offer credits/refunds.",
        system="You are a human expert with empathy and full authority to resolve issues."
    )

def support_pipeline(customer: str, issue: str) -> SupportSession:
    'Full tiered support pipeline with HITL escalation.'
    session = SupportSession(customer=customer, issue=issue)

    print(f"\n{'='*55}")
    print(f"Support Session: {session.session_id} | Customer: {customer}")
    print(f"Issue: {issue[:80]}")

    tier, reason = classify_support_issue(issue)
    session.tier = tier
    session.escalation_reason = reason
    print(f"\n🔀 Classified: {tier.value.upper()} | {reason}")

    if tier == SupportTier.TIER1:
        print("  🤖 AI Chatbot handling...")
        response = tier1_handler(session)
    elif tier == SupportTier.TIER2:
        print("  🔧 AI + Tools handling...")
        response = tier2_handler(session)
    else:
        print("  👤 Escalating to human specialist...")
        response = tier3_human_handler(session)

    session.history.append({"role": tier.value, "response": response})
    session.resolved = True
    print(f"\n✅ Response:\n{response[:300]}")
    return session

print("=== TIERED SUPPORT HITL PIPELINE ===")
test_cases = [
    ("Alice", "How do I reset my password?"),
    ("Bob",   "I was charged twice for my subscription this month."),
    ("Carol", "I believe my account was hacked and my private data has been leaked. I'm considering legal action."),
]

sessions = [support_pipeline(c, i) for c, i in test_cases]
print(f"\n📊 Processed {len(sessions)} sessions | "
      f"Tier distribution: {dict(zip(*[[t.value for t in SupportTier],[sum(1 for s in sessions if s.tier==t) for t in SupportTier]]))}")

## 5. Use Case 3 — Financial Fraud Detection with Human Analyst

In [ ]:
@dataclass
class Transaction:
    txn_id:   str
    amount:   float
    merchant: str
    country:  str
    time:     str
    customer: str
    history:  str

def ai_fraud_check(txn: Transaction) -> Tuple[str, float, List[str]]:
    'AI fraud scoring. Returns (verdict, risk_score, flags).'
    raw = call_llm(
        f"""Evaluate this transaction for fraud risk.

Transaction:
- ID: {txn.txn_id}
- Amount: ${txn.amount:,.2f}
- Merchant: {txn.merchant}
- Country: {txn.country}
- Time: {txn.time}
- Customer history: {txn.history}

Return ONLY JSON:
{{"verdict": "approve"|"flag"|"block", "risk_score": <0.0-1.0>,
  "flags": ["<flag1>", "<flag2>"], "reason": "<brief>"}}""",
        system="You are an AI fraud detection system. Be precise. Flag unusual patterns."
    )
    try:
        data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
        return data["verdict"], float(data["risk_score"]), data.get("flags", [])
    except:
        return "flag", 0.5, ["parse_error"]

def human_analyst_review(txn: Transaction, ai_verdict: str,
                          risk_score: float, flags: List[str]) -> Tuple[str, str]:
    'Human fraud analyst reviews flagged transactions.'
    decision = call_llm(
        f"""You are a senior fraud analyst. Review this AI-flagged transaction.

Transaction: ${txn.amount:,.2f} at {txn.merchant} ({txn.country}) — {txn.time}
Customer: {txn.customer} | History: {txn.history}
AI verdict: {ai_verdict} | Risk score: {risk_score:.0%}
AI flags: {flags}

Make final decision: approve, block, or contact_customer.
Return ONLY JSON: {{"verdict": "approve"|"block"|"contact_customer", "notes": "<reasoning>"}}""",
        system="You are a senior human fraud analyst with full authority."
    )
    try:
        data = json.loads(re.sub(r"```[\w]*", "", decision).strip())
        return data["verdict"], data["notes"]
    except:
        return "block", "Parse error — blocking to be safe"

def fraud_pipeline(transactions: List[Transaction]) -> dict:
    stats = {"auto_approved": 0, "auto_blocked": 0, "human_reviewed": 0}

    for txn in transactions:
        print(f"\n💳 {txn.txn_id}: ${txn.amount:,.2f} at {txn.merchant} ({txn.country})")
        verdict, risk, flags = ai_fraud_check(txn)
        print(f"  AI: {verdict} | risk={risk:.0%} | flags={flags}")

        if verdict == "approve" and risk < 0.3:
            print(f"  ✅ Auto-approved")
            stats["auto_approved"] += 1
        elif verdict == "block" and risk > 0.85:
            print(f"  🔴 Auto-blocked")
            stats["auto_blocked"] += 1
        else:
            print(f"  🔍 Sending to human analyst...")
            human_verdict, notes = human_analyst_review(txn, verdict, risk, flags)
            stats["human_reviewed"] += 1
            print(f"  👤 Human: {human_verdict} | {notes[:100]}")

    return stats

print("=== FRAUD DETECTION HITL PIPELINE ===")
transactions = [
    Transaction("TXN-001", 45.00,     "Starbucks",        "Nigeria", "09:15 AM Mon",  "Alice",  "Regular $20-50 local purchases"),
    Transaction("TXN-002", 4850.00,   "Electronics Store","Romania", "03:22 AM Sat",  "Bob",    "Never purchases abroad, avg $80/txn"),
    Transaction("TXN-003", 1200.00,   "Airlines",         "UAE",     "11:45 AM Fri",  "Carol",  "Frequent traveller, regular large purchases"),
    Transaction("TXN-004", 9.99,      "Netflix",           "Nigeria", "12:01 AM Mon",  "David",  "Monthly subscription, same amount always"),
]

stats = fraud_pipeline(transactions)
print(f"\n📊 {stats}")

## 6. Use Case 4 — Human-on-the-Loop (Policy-Driven Autonomy)

In [ ]:
class HumanOnTheLoop:
    "'Human-on-the-loop' variant:"
    def __init__(self, policies: List[str], domain: str):
        self.policies  = policies
        self.domain    = domain
        self.actions:  List[Dict] = []
        self.escalated: List[Dict] = []

    def policy_text(self) -> str:
        return "\n".join(f"- {p}" for p in self.policies)

    def act(self, situation: str) -> dict:
        'AI checks if the situation is covered by policy, then acts.'
        raw = call_llm(
            f"""You are an AI agent operating under these human-defined policies.

POLICIES:
{self.policy_text()}

SITUATION: {situation}

1. Determine if this situation is covered by any policy.
2. If yes, execute autonomously.
3. If not covered or ambiguous, request human guidance.

Return ONLY JSON:
{{"covered_by_policy": true|false, "policy_applied": "<policy text or null>",
  "action": "<action taken or null>", "needs_human": true|false,
  "question_for_human": "<question or null>"}}""",
            system=f"You are an autonomous {self.domain} agent. Act within policy, escalate outside it."
        )
        try:
            result = json.loads(re.sub(r"```[\w]*", "", raw).strip())
        except:
            result = {"covered_by_policy": False, "needs_human": True,
                      "question_for_human": "Parse error — need human guidance"}

        if not isinstance(result, dict):
            result = {"covered_by_policy": False, "needs_human": True,
                      "question_for_human": "Invalid response format — need human guidance"}
        result["situation"] = situation

        if result.get("needs_human") is not False or result.get("covered_by_policy") is not True:
            self.escalated.append(result)
            print(f"  🙋 Needs human: {result.get('question_for_human', 'N/A')}")
        else:
            self.actions.append(result)
            print(f"  ✅ Autonomous action: {str(result.get('action') or 'No action required')[:100]}")

        return result

    def summary(self):
        total = len(self.actions) + len(self.escalated)
        print(f"\n📊 Human-on-the-Loop: {total} situations | "
              f"{len(self.actions)} autonomous | {len(self.escalated)} needed human")

print("=== HUMAN-ON-THE-LOOP: TRADING BOT ===")
trading_bot = HumanOnTheLoop(
    domain="investment",
    policies=[
        "Maintain 70% tech stocks, 30% bonds",
        "Never invest more than 5% in any single company",
        "Automatically sell any stock that falls 10% below purchase price",
        "Do not trade during market hours in a country experiencing political crisis",
        "Rebalance portfolio every Monday morning to maintain target allocation",
    ]
)

situations = [
    "AAPL has dropped 12% since purchase price.",
    "Portfolio is now 80% tech, 20% bonds after today's market movement.",
    "An IPO for a new quantum computing company (QuantumCo) opens tomorrow at $45/share.",
    "TSLA fell 8% today due to CEO tweet. Should we sell?",
    "Market is closed today for a public holiday.",
]

for situation in situations:
    print(f"\n📊 Situation: {situation}")
    trading_bot.act(situation)

trading_bot.summary()

print("\n=== HUMAN-ON-THE-LOOP: CALL CENTER ===")
call_center = HumanOnTheLoop(
    domain="customer service",
    policies=[
        "Route any mention of 'service outage' immediately to technical support",
        "If customer tone is highly frustrated, offer immediate transfer to senior agent",
        "Offer 10% discount to any customer who has been waiting over 20 minutes",
        "Never discuss competitor pricing",
        "Always verify customer identity before discussing account details",
    ]
)

calls = [
    "Customer says: 'I've been on hold for 25 minutes and I'm extremely angry!'",
    "Customer asks about our data breach that was reported on the news today.",
    "Customer asks: 'My internet has been down since this morning — is there an outage?'",
    "Customer wants to compare our prices with a competitor's offer.",
]

for call in calls:
    print(f"\n📞 Call: {call[:70]}")
    call_center.act(call)

call_center.summary()

## 7. Use Case 5 — RLHF Data Collection Loop

In [ ]:
class RLHFDataCollector:
    'Collects human preference data for RLHF fine-tuning.'
    def __init__(self, n_responses: int = 3):
        self.n_responses = n_responses
        self.dataset:    List[Dict] = []

    def generate_responses(self, prompt: str) -> List[str]:
        'Generate N diverse responses using different temperatures/personas.'
        personas = [
            "Be concise and direct",
            "Be detailed and thorough",
            "Be friendly and conversational",
        ]
        responses = []
        for persona in personas[:self.n_responses]:
            r = call_llm(prompt, system=f"You are a helpful assistant. {persona}.")
            responses.append(r)
        return responses

    def simulated_human_rank(self, prompt: str, responses: List[str]) -> Tuple[int, str]:
        'Simulates a human ranking responses (returns index of best + reasoning).'
        numbered = "\n\n".join(f"Response {i+1}:\n{r}" for i, r in enumerate(responses))
        raw = call_llm(
            f"""You are a human evaluator comparing AI responses.
Pick the BEST response based on: helpfulness, accuracy, clarity, safety.

Prompt: {prompt}

{numbered}

Return ONLY JSON: {{"best": <1|2|3>, "worst": <1|2|3>, "reason": "<brief>"}}""",
            system="You are a careful human evaluator. Return only valid JSON."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", raw).strip())
            return int(data["best"]) - 1, int(data["worst"]) - 1, data["reason"]
        except:
            return 0, 1, "Parse error"

    def collect(self, prompts: List[str]):
        'Run full RLHF collection loop.'
        for prompt in prompts:
            print(f"\n📝 Prompt: {prompt[:70]}")

            responses = self.generate_responses(prompt)
            print(f"  Generated {len(responses)} candidate responses")

            best_idx, worst_idx, reason = self.simulated_human_rank(prompt, responses)
            print(f"  👤 Human preferred response {best_idx+1} over {worst_idx+1}: {reason}")

            self.dataset.append({
                "prompt":   prompt,
                "chosen":   responses[best_idx],
                "rejected": responses[worst_idx],
                "reason":   reason
            })

        print(f"\n✅ RLHF dataset: {len(self.dataset)} preference pairs collected")
        print(f"   Format: {{prompt, chosen, rejected}} — ready for DPO fine-tuning")

    def export_jsonl(self) -> str:
        'Export dataset as JSONL string (ready for fine-tuning).'
        return "\n".join(json.dumps(row) for row in self.dataset)

print("=== RLHF DATA COLLECTION LOOP ===")
collector = RLHFDataCollector(n_responses=3)

training_prompts = [
    "Explain what a neural network is to a 10-year-old.",
    "How should I handle a conflict with my manager at work?",
    "What are the pros and cons of remote work?",
]

collector.collect(training_prompts)

print("\n📄 Sample JSONL output (first entry preview):")
if collector.dataset:
    sample = collector.dataset[0]
    print(f"  prompt:   {sample['prompt'][:60]}")
    print(f"  chosen:   {sample['chosen'][:80]}...")
    print(f"  rejected: {sample['rejected'][:80]}...")

## 8. Approval Gate — Blocking Actions Until Human Approves

In [ ]:
class ApprovalGate:
    'Blocks an agent action until a human explicitly approves it.'
    def __init__(self, auto_approve_threshold: float = 0.95):
        self.threshold = auto_approve_threshold
        self.pending:  List[Dict] = []
        self.approved: List[Dict] = []
        self.rejected: List[Dict] = []

    def request_approval(self, action: str, risk_level: str,
                         justification: str, estimated_impact: str) -> Dict:
        'Submit an action for human approval.'
        request = {
            "id":              f"APPR-{uuid.uuid4().hex[:6].upper()}",
            "action":          action,
            "risk_level":      risk_level,
            "justification":   justification,
            "estimated_impact": estimated_impact,
            "status":          "pending",
            "submitted_at":    datetime.now().isoformat()[:19]
        }
        self.pending.append(request)

        print(f"\n⏸️  APPROVAL GATE: {request['id']}")
        print(f"   Action:    {action}")
        print(f"   Risk:      {risk_level.upper()}")
        print(f"   Impact:    {estimated_impact}")
        print(f"   Why:       {justification}")

        if risk_level in ("high", "critical"):
            return self._simulate_human_decision(request)
        else:

            request["status"] = "auto_approved"
            self.pending.remove(request)
            self.approved.append(request)
            print(f"   ✅ Auto-approved (low risk)")
            return request

    def _simulate_human_decision(self, request: Dict) -> Dict:
        'Simulates human reviewing and deciding.'
        decision = call_llm(
            f"""You are a human approver for an AI system.
Review this action request and decide to approve or reject.

Action: {request['action']}
Risk level: {request['risk_level']}
Justification: {request['justification']}
Impact: {request['estimated_impact']}

Return ONLY JSON: {{"approved": true|false, "conditions": "<any conditions or null>"}}""",
            system="You are a cautious human approver. Reject if risk outweighs benefit."
        )
        try:
            data = json.loads(re.sub(r"```[\w]*", "", decision).strip())
            approved = data.get("approved") is True
        except:
            approved = False

        request["status"] = "approved" if approved else "rejected"
        self.pending.remove(request)
        (self.approved if approved else self.rejected).append(request)

        icon = "✅" if approved else "❌"
        print(f"   {icon} Human decision: {'APPROVED' if approved else 'REJECTED'}")
        return request

    def summary(self):
        total = len(self.approved) + len(self.rejected) + len(self.pending)
        print(f"\n📊 Approval Gate: {total} requests | "
              f"{len(self.approved)} approved | {len(self.rejected)} rejected | "
              f"{len(self.pending)} pending")

print("=== APPROVAL GATE PATTERN ===")
gate = ApprovalGate()

actions = [
    ("Send welcome email to 50 new users",
     "low", "User signup automation", "50 emails sent"),
    ("Delete all records from the staging database",
     "high", "Monthly data cleanup process", "Irreversible — 10,000 records deleted"),
    ("Deploy new model to production API",
     "high", "Performance improvement validated on staging", "All API responses changed"),
    ("Transfer $250,000 to supplier account",
     "critical", "Approved invoice #INV-4421", "$250K wire transfer — irreversible"),
    ("Archive 90-day-old log files",
     "low", "Storage optimization", "~2GB disk space freed"),
]

for action, risk, justification, impact in actions:
    gate.request_approval(action, risk, justification, impact)

gate.summary()

## 9. Quick Reference

In [ ]:
reference = """
╔═══════════════════════════╦══════════════════════╦═════════════════════════════╗
║ HITL Mode                 ║ When to Use          ║ Implementation              ║
╠═══════════════════════════╬══════════════════════╬═════════════════════════════╣
║ Confidence-Gated          ║ General tasks with   ║ Score output → route by     ║
║ Escalation                ║ variable complexity  ║ ConfidenceLevel             ║
╠═══════════════════════════╬══════════════════════╬═════════════════════════════╣
║ Tiered Support            ║ Customer service,    ║ Classify tier → route to    ║
║                           ║ technical support    ║ AI/AI+tools/human           ║
╠═══════════════════════════╬══════════════════════╬═════════════════════════════╣
║ Human-on-the-Loop         ║ High-speed decisions ║ Human sets policies →        ║
║ (Policy-Driven)           ║ with known rules     ║ AI executes autonomously    ║
╠═══════════════════════════╬══════════════════════╬═════════════════════════════╣
║ Approval Gate             ║ Irreversible/high-   ║ Block action → require      ║
║                           ║ stakes actions       ║ human approve/reject        ║
╠═══════════════════════════╬══════════════════════╬═════════════════════════════╣
║ RLHF Collection           ║ Model improvement    ║ Generate N responses →      ║
║                           ║ preference learning  ║ human ranks → DPO dataset   ║
╠═══════════════════════════╬══════════════════════╬═════════════════════════════╣
║ Moderation Pipeline       ║ Content platforms    ║ AI first pass → escalate    ║
║                           ║                      ║ ambiguous to human          ║
╚═══════════════════════════╩══════════════════════╩═════════════════════════════╝

HITL TRADE-OFFS:
  ✅ Higher accuracy    ✅ Ethical alignment    ✅ Builds trust
  ❌ Doesn't scale      ❌ Needs expert humans  ❌ Privacy risks with sensitive data

RULE: Auto-handle HIGH confidence. Flag MEDIUM. Block on LOW. Escalate on NONE.
"""
print(reference)

## 10. Exercises

- **Exercise 1 — Dynamic Threshold:** Modify `HITLAgent` so the confidence threshold adjusts dynamically. After 10 interactions, if human review agrees with AI output >90% of the time, raise the threshold by 0.05 (trust the AI more). If agreement drops below 70%, lower it by 0.05.

- **Exercise 2 — Async Approval:** Refactor `ApprovalGate` to use `asyncio`. High-risk actions are submitted to the queue but don't block — the agent continues other tasks and re-checks the approval status every 5 seconds.

- **Exercise 3 — Bias Detection:** Add a `BiasReviewer` to the content moderation pipeline. Before a human sees an escalated item, a second AI agent checks the first AI's reasoning for demographic bias (e.g. treating similar content differently based on implied group identity) and flags it.

- **Exercise 4 — RLHF Quality Filter:** Extend `RLHFDataCollector` to reject training pairs where the difference between chosen and rejected responses is too small (similarity score > 0.85 using TF-IDF cosine similarity). Only high-contrast pairs enter the dataset.

- **Exercise 5 — Human-on-the-Loop with Policy Learning:** When `HumanOnTheLoop` escalates a situation and receives human guidance, automatically extract a new policy rule from that guidance and add it to `self.policies` so similar future situations are handled autonomously.

**Next:** Chapter 14 — RAG (Retrieval-Augmented Generation)